# Dashboard SpaceX — version Google Colab

Ce notebook contient **exactement le même dashboard** que `spacex_dash_app.py`, mais adapté pour tourner directement dans Google Colab (le fichier `.py` seul ne s'affiche pas dans un navigateur depuis Colab sans ce petit ajustement).

**Marche à suivre :**
1. Exécute toutes les cellules dans l'ordre
2. Une fenêtre/onglet va s'ouvrir avec ton dashboard interactif
3. Prends des captures d'écran (demandées dans les consignes) :
   - Vue "All Sites" (pie chart)
   - Vue d'un site spécifique
   - Le scatter plot avec le slider de payload ajusté
4. Garde ce notebook complété + les captures pour ton dépôt GitHub

In [ ]:
!pip install dash pandas -q

In [ ]:
# Import des librairies
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px
import threading

# Chargement des données
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()
launch_sites = spacex_df['Launch Site'].unique().tolist()

spacex_df.head()

In [ ]:
# Construction de l'application Dash (identique à spacex_dash_app.py)
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),

    dcc.Dropdown(
        id='site-dropdown',
        options=[{'label': 'All Sites', 'value': 'ALL'}] +
                [{'label': site, 'value': site} for site in launch_sites],
        value='ALL',
        placeholder="Select a Launch Site here",
        searchable=True
    ),
    html.Br(),

    html.Div(dcc.Graph(id='success-pie-chart')),
    html.Br(),

    html.P("Payload range (Kg):"),
    dcc.RangeSlider(
        id='payload-slider',
        min=0, max=10000, step=1000,
        marks={0: '0', 2500: '2500', 5000: '5000', 7500: '7500', 10000: '10000'},
        value=[min_payload, max_payload]
    ),

    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])


@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):
    filtered_df = spacex_df
    if entered_site == 'ALL':
        fig = px.pie(filtered_df, values='class', names='Launch Site',
                     title='Total Success Launches By Site')
        return fig
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        outcome_counts = site_df['class'].value_counts().reset_index()
        outcome_counts.columns = ['class', 'count']
        outcome_counts['class'] = outcome_counts['class'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(outcome_counts, values='count', names='class',
                     title=f'Total Success Launches for site {entered_site}')
        return fig


@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [Input(component_id='site-dropdown', component_property='value'),
     Input(component_id="payload-slider", component_property="value")]
)
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    mask = (spacex_df['Payload Mass (kg)'] >= low) & (spacex_df['Payload Mass (kg)'] <= high)
    filtered_df = spacex_df[mask]
    if entered_site == 'ALL':
        fig = px.scatter(filtered_df, x='Payload Mass (kg)', y='class',
                          color='Booster Version Category',
                          title='Correlation between Payload and Success for all Sites')
        return fig
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        fig = px.scatter(site_df, x='Payload Mass (kg)', y='class',
                          color='Booster Version Category',
                          title=f'Correlation between Payload and Success for site {entered_site}')
        return fig

print("App Dash construite avec succes.")

## Lancer le dashboard

La cellule suivante démarre le serveur Dash en arrière-plan, puis ouvre une fenêtre pointant dessus grâce à `google.colab.output` (le système de Colab qui redirige un port local vers ton navigateur).

Si une erreur `ModuleNotFoundError: No module named 'google.colab'` apparaît, c'est que tu n'es pas sur Colab (par ex. Jupyter local) : dans ce cas, remplace la dernière ligne par `app.run(debug=True)` et ouvre `http://127.0.0.1:8050` toi-même.

In [ ]:
def run_app():
    app.run(port=8050)

threading.Thread(target=run_app, daemon=True).start()

try:
    from google.colab import output
    output.serve_kernel_port_as_window(8050)
except ImportError:
    print("Pas sur Google Colab : ouvre http://127.0.0.1:8050 dans ton navigateur.")

---
### ✅ Après avoir exploré le dashboard

Réponds à ces 5 questions (demandées dans les consignes) en te basant sur ce que tu observes :

1. Quel site a le plus de lancements réussis ?
2. Quel site a le meilleur taux de réussite ?
3. Quelle plage de payload a le meilleur taux de réussite ?
4. Quelle plage de payload a le pire taux de réussite ?
5. Quelle version de booster F9 (v1.0, v1.1, FT, B4, B5...) a le meilleur taux de réussite ?

Note tes réponses quelque part — elles serviront directement pour le slide du dashboard dans ton PowerPoint.